# Attention-Only Ablation Notebook

这个 notebook 用于在 **不修改残差结构和 FFN 结构** 的前提下，只比较注意力模块：

- `baseline`：标准 decoder-only Transformer
- `shared_kv_depth_memory`：层独立 `W_Q^{(l)}`，跨层共享 `W_K/W_V`，当前 token 额外读取“同位置跨层历史”的 attention 变体

特性：

- **单文件 notebook**
- **不依赖当前仓库中的任何自定义 Python 模块**
- **数据直接通过 Hugging Face `datasets` 加载**
- **SwanLab 记录默认写入新项目**
- **支持用户在配置区修改模型大小、数据集、训练步数**
- **支持多 GPU（Kaggle 若分配到两张 T4，会自动尝试 `DataParallel`）**

默认实现遵循本次实验约束：

1. 只修改 attention 模块  
2. residual / MLP / 训练配方在 baseline 和自定义方法之间保持一致  
3. 自定义方法采用：
   - 层独立 `W_Q^{(l)}`
   - 跨层共享 `W_K, W_V`
   - 历史 memory 为“同 token 位置的跨层历史”
4. 训练过程中验证使用**顺序游标评估**，避免每次只盯住验证集开头同一小段  
5. 最终测试默认使用**全量 test split**（可改成更大的固定批次数）


In [1]:
# 安装依赖（Kaggle 首次运行时执行）
!pip -q install datasets transformers swanlab
import os
os.environ["SWANLAB_API_KEY"] = "TaeDjfuP66mvYqL9hVn33"


In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "src")))
from layer_depth_attention.ablation_models import TinyDecoderLM

In [3]:
# 设置swanlab监控类
import swanlab


class SwanLabMonitor:
    def __init__(self, backend: str, project: str, experiment_name: str, config: dict):
        self.backend = backend
        self.project = project
        self.experiment_name = experiment_name
        self.config = config
        self.enabled = backend == "swanlab" and swanlab is not None

    def init(self):
        if not self.enabled:
            print("[swanlab] disabled")
            return
        try:
            api_key = os.environ.get("SWANLAB_API_KEY")
            if api_key:
                swanlab.login(api_key=api_key)
            else:
                swanlab.login()
            swanlab.init(project=self.project, experiment_name=self.experiment_name, config=self.config)
            print(f"[swanlab] init succeeded: project={self.project} experiment={self.experiment_name}")
        except Exception as exc:
            self.enabled = False
            print(f"[swanlab] init failed: {exc!r}")

    def log(self, metrics: dict, step: int):
        if self.enabled:
            try:
                swanlab.log(metrics, step=step)
            except Exception as exc:
                print(f"[swanlab] log failed: {exc!r}")

    def finish(self):
        if self.enabled:
            try:
                swanlab.finish()
            except Exception as exc:
                print(f"[swanlab] finish failed: {exc!r}")



In [4]:
import json
import math
import os
import random
import time
from collections import deque
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset

# from transformers import GPT2Tokenizer
from transformers import GPT2Tokenizer


try:
    import swanlab
except Exception:
    swanlab = None



SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("device =", DEVICE, "num_gpus =", NUM_GPUS)


device = cuda num_gpus = 1


In [5]:
# # =========================
# # 可改参数区
# # =========================

# CFG = {
#     # 数据
#     "dataset_name": "wikitext",
#     "dataset_config": "wikitext-2-raw-v1",   # 也可改成 wikitext-103-raw-v1
#     "text_field": "text",
#     "tokenizer_name": "gpt2",
#     "add_eos_between_lines": True,

#     # 模型
#     "max_seq_len": 256,
#     "d_model": 384,
#     "num_layers": 8,
#     "num_heads": 8,
#     "mlp_ratio": 4,
#     "dropout": 0.1,
#     "tie_weights": True,
#     "use_pos_emb": True,

#     # 训练
#     "batch_size": 16,
#     "grad_accum_steps": 1,
#     "steps": 40000,
#     "eval_interval": 200,
#     "eval_batches": 20,            # 训练中验证采用顺序游标评估
#     "final_test_batches": None,    # None 表示全量 test；也可改成整数
#     "train_loss_window": 20,
#     "lr": 3e-4,
#     "min_lr_scale": 0.1,
#     "warmup_steps": 100,
#     "weight_decay": 0.01,
#     "grad_clip": 1.0,

#     # 运行
#     "methods_to_run": ["shared_kv_baseline"],
#     "use_data_parallel": True,
#     "output_dir": "/kaggle/working/attention_only_ablation_outputs",

#     # SwanLab（不要使用之前项目）
#     "log_backend": "swanlab",  # "none" 或 "swanlab"
#     "log_project": "Layer-Depth-Attention-Kaggle",
#     "log_workspace": "justbook",
#     "run_note": "attention_only_ablation_kaggle",
# }

# Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
# print(json.dumps(CFG, indent=2, ensure_ascii=False))


In [6]:
# =========================
# 可改参数区 (Wikitext-103 定制最终版)
# =========================

CFG = {
    # 数据 (切换到了 1.03 亿词汇的真正大数据集)
    "dataset_name": "wikitext",
    "dataset_config": "wikitext-103-raw-v1",   
    "text_field": "text",
    "tokenizer_name": "gpt2",
    "add_eos_between_lines": True,
    "data_source": "hf",            # 必须加上这个，说明从 Hugging Face 读取
    "local_data_dir": "",  
    # 模型
    "max_seq_len": 256,
    "d_model": 384,
    "num_layers": 8,
    "num_heads": 8,
    "mlp_ratio": 4,
    "dropout": 0.1,
    "tie_weights": True,
    "use_pos_emb": True,

    # 训练 (增加步数防止过早没收敛)
    "batch_size": 8,
    "grad_accum_steps": 1,
    "steps": 40000,
    "eval_interval": 1000,
    "eval_batches": 100,            # 每次游标看 100 批验证数据
    "final_test_batches": None,     # None 表示考完全局 test 测试卷
    "train_loss_window": 20,
    "lr": 3e-4,
    "min_lr_scale": 0.1,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "grad_clip": 1.0,

    # 运行 (这里把修好双轴方法和对照组一起跑)
    "methods_to_run": [
        # "baseline", 
        # "shared_kv_baseline", 
        # "shared_kv_depth_memory_dualq"，
        "shared_kv_depth_memory_dualq_sublayer" 
        # "attn_residual"
        # "attn_residual_2d"
    ],
    "use_data_parallel": True,
    "output_dir": "/kaggle/working/attention_only_ablation_outputs",

    # SwanLab
    "log_backend": "swanlab",  
    "log_project": "Layer-Depth-Attention-Kaggle-103",
    "log_workspace": "justbook", 
    "run_note": "attention_wikitext103_final",
}


In [7]:
import hashlib
import array
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Iterator, Optional
import math
import torch

@dataclass
class LMCfg:
    data_source: str
    local_data_dir: str
    dataset_name: str
    dataset_config: str
    text_field: str
    tokenizer_name: str
    max_seq_len: int
    add_eos_between_lines: bool = True

class LMData:
    def __init__(self, cfg: LMCfg):
        self.cfg = cfg
        self.tokenizer = GPT2Tokenizer.from_pretrained(cfg.tokenizer_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.model_max_length = 100000000  # 消除长文本警告
        self.vocab_size = self.tokenizer.vocab_size
        self.pad_token_id = self.tokenizer.eos_token_id
        self.dataset = None
        self.local_data_dir = Path(cfg.local_data_dir) if cfg.local_data_dir else Path(".")
        if cfg.data_source == "hf":
            self.dataset = load_dataset(cfg.dataset_name, cfg.dataset_config)

        # ✅ 有缓存直接加载，没有就分词后保存
        self.splits = self._load_or_tokenize()

        self.eval_windows = {
            split: self._build_eval_windows(self.splits[split])
            for split in ["validation", "test"]
        }
        self.eval_cursors = {"validation": 0, "test": 0}

    # -------------------------------------------------------
    # 缓存逻辑
    # -------------------------------------------------------
    def _cache_path(self) -> Path:
        """根据数据集+分词器配置生成唯一缓存路径"""
        key = (
            f"{self.cfg.dataset_name}_{self.cfg.dataset_config}"
            f"_{self.cfg.tokenizer_name}_{self.cfg.add_eos_between_lines}"
        )
        hash_str = hashlib.md5(key.encode()).hexdigest()[:10]
        cache_dir = Path(".cache/lmdata")
        cache_dir.mkdir(parents=True, exist_ok=True)
        return cache_dir / f"{hash_str}.pt"

    def _load_or_tokenize(self) -> Dict[str, torch.Tensor]:
        cache_file = self._cache_path()
        if cache_file.exists():
            print(f"⚡ 发现缓存，直接加载: {cache_file}")
            splits = torch.load(cache_file, weights_only=True)
            print("✅ 缓存加载完成！")
            return splits

        print("🔄 未发现缓存，开始分词（首次需要几分钟，之后秒开）...")
        splits = self._tokenize_all_splits()
        torch.save(splits, cache_file)
        print(f"💾 已保存缓存到: {cache_file}")
        return splits

    # -------------------------------------------------------
    # 分词逻辑
    # -------------------------------------------------------
    def _tokenize_split(self, split_name: str) -> torch.Tensor:
        if self.cfg.data_source == "local_text":
            text_path = self.local_data_dir / f"{split_name}.txt"
            texts = text_path.read_text(encoding="utf-8").split("\n")
        else:
            texts = self.dataset[split_name][self.cfg.text_field]

        all_ids = []
        eos_id = self.tokenizer.eos_token_id
        for text in texts:
            if len(text.strip()) > 0:
                ids = self.tokenizer.encode(text, add_special_tokens=False, truncation=False)
                all_ids.extend(ids)
            if self.cfg.add_eos_between_lines:
                all_ids.append(eos_id)

        return torch.tensor(all_ids, dtype=torch.long)

    def _tokenize_all_splits(self) -> Dict[str, torch.Tensor]:
        return {
            "train":      self._tokenize_split("train"),
            "validation": self._tokenize_split("validation"),
            "test":       self._tokenize_split("test"),
        }

    # -------------------------------------------------------
    # 批次采样
    # -------------------------------------------------------
    def _build_eval_windows(self, token_ids: torch.Tensor) -> List[Tuple[torch.Tensor, torch.Tensor]]:
        chunk = self.cfg.max_seq_len + 1
        max_offset = token_ids.numel() - chunk
        starts = list(range(0, max_offset + 1, self.cfg.max_seq_len))
        windows = []
        for start in starts:
            piece = token_ids[start : start + chunk]
            windows.append((piece[:-1], piece[1:]))
        return windows

    def sample_train_batch(self, batch_size: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
        token_ids = self.splits["train"]
        chunk = self.cfg.max_seq_len + 1
        max_start = token_ids.numel() - chunk
        starts = torch.randint(0, max_start + 1, (batch_size,))
        windows = [token_ids[s : s + chunk] for s in starts.tolist()]
        batch = torch.stack(windows, dim=0).to(device)
        return batch[:, :-1], batch[:, 1:]

    def iter_eval_batches(
        self,
        split: str,
        batch_size: int,
        device: torch.device,
        max_batches: Optional[int],
        use_cursor: bool,
    ) -> Iterator[Tuple[torch.Tensor, torch.Tensor]]:
        windows = self.eval_windows[split]
        total = len(windows)
        if total == 0:
            return
        batches_to_yield = math.ceil(total / batch_size) if max_batches is None else max_batches
        cursor = self.eval_cursors[split] if use_cursor else 0
        for _ in range(batches_to_yield):
            current = []
            for _ in range(batch_size):
                current.append(windows[cursor])
                cursor = (cursor + 1) % total
                if max_batches is None and cursor == 0 and len(current) > 0:
                    break
            xs = torch.stack([item[0] for item in current], dim=0).to(device)
            ys = torch.stack([item[1] for item in current], dim=0).to(device)
            yield xs, ys
            if max_batches is None and cursor == 0:
                break
        if use_cursor:
            self.eval_cursors[split] = cursor


# -------------------------------------------------------
# 初始化数据（有缓存秒开，无缓存首次等几分钟）
# -------------------------------------------------------
data = LMData(
    LMCfg(
        data_source=CFG["data_source"],
        local_data_dir=CFG["local_data_dir"],
        dataset_name=CFG["dataset_name"],
        dataset_config=CFG["dataset_config"],
        text_field=CFG["text_field"],
        tokenizer_name=CFG["tokenizer_name"],
        max_seq_len=CFG["max_seq_len"],
        add_eos_between_lines=CFG["add_eos_between_lines"],
    )
)

print("vocab_size =", data.vocab_size)
for split_name, ids in data.splits.items():
    print(split_name, "tokens =", ids.numel())


⚡ 发现缓存，直接加载: .cache\lmdata\b98d9d371a.pt
✅ 缓存加载完成！
vocab_size = 50257
train tokens = 119721490
validation tokens = 251049
test tokens = 287645


In [8]:
def build_optimizer(model: nn.Module, lr: float, weight_decay: float):
    decay_params, nodecay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.dim() >= 2 and "norm" not in name.lower():
            decay_params.append(param)
        else:
            nodecay_params.append(param)
    return torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ],
        lr=lr,
        betas=(0.9, 0.95),
    )


def cosine_lr(step: int, total_steps: int, base_lr: float, warmup_steps: int, min_lr_scale: float) -> float:
    if step <= warmup_steps:
        return base_lr * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return base_lr * (min_lr_scale + (1.0 - min_lr_scale) * cosine)


class SwanLabMonitor:
    def __init__(self, backend: str, project: str, experiment_name: str, config: dict):
        self.backend = backend
        self.project = project
        self.experiment_name = experiment_name
        self.config = config
        self.enabled = backend == "swanlab" and swanlab is not None

    def init(self):
        if not self.enabled:
            print("[swanlab] disabled")
            return
        try:
            api_key = os.environ.get("SWANLAB_API_KEY")
            if api_key:
                swanlab.login(api_key=api_key)
            else:
                swanlab.login()
            swanlab.init(project=self.project, experiment_name=self.experiment_name, config=self.config)
            print(f"[swanlab] init succeeded: project={self.project} experiment={self.experiment_name}")
        except Exception as exc:
            self.enabled = False
            print(f"[swanlab] init failed: {exc!r}")

    def log(self, metrics: dict, step: int):
        if self.enabled:
            try:
                swanlab.log(metrics, step=step)
            except Exception as exc:
                print(f"[swanlab] log failed: {exc!r}")

    def finish(self):
        if self.enabled:
            try:
                swanlab.finish()
            except Exception as exc:
                print(f"[swanlab] finish failed: {exc!r}")


@torch.no_grad()
def evaluate(model, data, split, batch_size, eval_batches, device, use_cursor):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    total_batches = 0
    for inputs, labels in data.iter_eval_batches(
        split=split,
        batch_size=batch_size,
        device=device,
        max_batches=eval_batches,
        use_cursor=use_cursor,
    ):
        logits = model(inputs)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        total_loss += loss.item() * labels.numel()
        total_tokens += labels.numel()
        total_batches += 1
    model.train()
    mean_loss = total_loss / max(total_tokens, 1)
    return {
        "loss": mean_loss,
        "perplexity": math.exp(mean_loss),
        "batches": total_batches,
    }


def run_experiment(method_name: str, cfg: dict):
    device = torch.device(DEVICE)
    model = TinyDecoderLM(
        vocab_size=data.vocab_size,
        max_seq_len=cfg["max_seq_len"],
        d_model=cfg["d_model"],
        num_layers=cfg["num_layers"],
        num_heads=cfg["num_heads"],
        mlp_ratio=cfg["mlp_ratio"],
        dropout=cfg["dropout"],
        attention_type=method_name,
        tie_weights=cfg["tie_weights"],
        use_pos_emb=cfg["use_pos_emb"],
    )
    if device.type == "cuda" and cfg["use_data_parallel"] and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    param_count = sum(p.numel() for p in model.parameters())

    optimizer = build_optimizer(model, cfg["lr"], cfg["weight_decay"])
    history = []
    recent_train_losses = deque(maxlen=cfg["train_loss_window"])
    best_record = None
    best_state = None
    started = time.perf_counter()

    monitor = SwanLabMonitor(
        backend=cfg["log_backend"],
        project=cfg["log_project"],
        experiment_name=f"{method_name}_{cfg['dataset_config']}_{cfg['d_model']}d_{cfg['num_layers']}l",
        config={**cfg, "method_name": method_name, "vocab_size": data.vocab_size, "model_params": param_count},
    )
    monitor.init()
    monitor.log({"model_params": param_count}, step=0)

    for step in range(1, cfg["steps"] + 1):
        lr = cosine_lr(step, cfg["steps"], cfg["lr"], cfg["warmup_steps"], cfg["min_lr_scale"])
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)

        step_loss_sum = 0.0
        for _ in range(cfg["grad_accum_steps"]):
            inputs, labels = data.sample_train_batch(cfg["batch_size"], device)
            logits = model(inputs)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
            (loss / cfg["grad_accum_steps"]).backward()
            step_loss_sum += loss.item()

        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        optimizer.step()

        train_loss = step_loss_sum / cfg["grad_accum_steps"]
        recent_train_losses.append(train_loss)

        if step % cfg["eval_interval"] == 0 or step == 1 or step == cfg["steps"]:
            val_metrics = evaluate(
                model=model,
                data=data,
                split="validation",
                batch_size=cfg["batch_size"],
                eval_batches=cfg["eval_batches"],
                device=device,
                use_cursor=True,
            )
            elapsed_seconds = time.perf_counter() - started
            record = {
                "step": step,
                "lr": lr,
                "train_loss_raw": train_loss,
                "train_loss_avg": sum(recent_train_losses) / len(recent_train_losses),
                "val_loss": val_metrics["loss"],
                "val_ppl": val_metrics["perplexity"],
                "elapsed_seconds": elapsed_seconds,
                "elapsed_minutes": elapsed_seconds / 60.0,
            }
            history.append(record)
            if best_record is None or record["val_loss"] < best_record["val_loss"]:
                best_record = dict(record)
                source_model = model.module if isinstance(model, nn.DataParallel) else model
                best_state = {k: v.detach().cpu().clone() for k, v in source_model.state_dict().items()}
            monitor.log(record, step=step)
            print(
                f"method={method_name} step={step} lr={lr:.6f} "
                f"train_loss_avg={record['train_loss_avg']:.4f} "
                f"val_loss={record['val_loss']:.4f} val_ppl={record['val_ppl']:.2f} "
                f"elapsed_min={record['elapsed_minutes']:.2f}"
            )

    source_model = model.module if isinstance(model, nn.DataParallel) else model
    if best_state is not None:
        source_model.load_state_dict(best_state)

    test_metrics = evaluate(
        model=model,
        data=data,
        split="test",
        batch_size=cfg["batch_size"],
        eval_batches=cfg["final_test_batches"],
        device=device,
        use_cursor=False,
    )

    summary = {
        "method": method_name,
        "model_params": param_count,
        "best_step": None if best_record is None else best_record["step"],
        "best_val_loss": None if best_record is None else best_record["val_loss"],
        "best_val_ppl": None if best_record is None else best_record["val_ppl"],
        "best_test_loss": test_metrics["loss"],
        "best_test_ppl": test_metrics["perplexity"],
        "history": history,
        "config": cfg,
    }

    out_path = Path(cfg["output_dir"]) / f"{method_name}_{cfg['dataset_config']}_{cfg['d_model']}d_{cfg['num_layers']}l.json"
    out_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    monitor.log({"best_test_loss": summary["best_test_loss"], "best_test_ppl": summary["best_test_ppl"]}, step=cfg["steps"])
        # 把成绩以极具仪式感的方式打印到屏幕上
    print("\n" + "=" * 60)
    print(f"🎉 实验大功告成: {method_name}")
    print(f"📦 模型参数量: {param_count / 1e6:.2f} M (百万)")
    print(f"🏆 最佳 验证集(Val) PPL:  {summary['best_val_ppl']:.2f} (出现在第 {summary['best_step']} 步)")
    print(f"🔥 最终 测试集(Test) PPL: {summary['best_test_ppl']:.2f}")
    print("=" * 60 + "\n")
    print(f"当前方法特点：删除残差")
    print(f"当前方法特点总结: {method_name}\n取消注意力层的残差连接，完全依赖注意力机制来传递信息，这可能会导致训练不稳定，尤其是在深层模型中。")
    monitor.finish()
    return summary

    monitor.finish()
    return summary


In [9]:
results = {}
for method_name in CFG["methods_to_run"]:
    print("=" * 80)
    print("running", method_name)
    print("=" * 80)
    results[method_name] = run_experiment(method_name, CFG)

results


running shared_kv_depth_memory_dualq_sublayer


Output()

swanlab: swanlab version 0.7.14 is available!  Upgrade: `pip install -U swanlab`

Output()

swanlab: Tracking run with swanlab version 0.6.13

...

swanlab: 🚀 View run at https://swanlab.cn/@justbook/Layer-Depth-Attention-Kaggle-103/runs/htv1zdoxfxfr5seokcrsc

[swanlab] init succeeded: project=Layer-Depth-Attention-Kaggle-103 experiment=shared_kv_depth_memory_dualq_sublayer_wikitext-103-raw-v1_384d_8l
method=shared_kv_depth_memory_dualq_sublayer step=1 lr=0.000003 train_loss_avg=10.8832 val_loss=10.8553 val_ppl=51807.87 elapsed_min=0.13
method=shared_kv_depth_memory_dualq_sublayer step=1000 lr=0.000300 train_loss_avg=6.0376 val_loss=6.0305 val_ppl=415.91 elapsed_min=2.08
method=shared_kv_depth_memory_dualq_sublayer step=2000 lr=0.000298 train_loss_avg=5.8446 val_loss=5.7008 val_ppl=299.12 elapsed_min=4.01
method=shared_kv_depth_memory_dualq_sublayer step=3000 lr=0.000296 train_loss_avg=5.5548 val_loss=5.4826 val_ppl=240.48 elapsed_min=5.94
method=shared_kv_depth_memory_dualq_sublayer step=4000 lr=0.000294 train_loss_avg=5.4233 val_loss=5.3642 val_ppl=213.63 elapsed_min=7.87
method=shared_kv_depth_memory_dualq_sublayer step=5000 lr=0.000290 train_loss_avg=5.4068 val_loss=5.2225 val_ppl=185.40 elapsed_min=9.80
method=shared_kv_depth_memory_dua

swanlab: 🏠 View project at https://swanlab.cn/@justbook/Layer-Depth-Attention-Kaggle-103

swanlab: 🚀 View run at https://swanlab.cn/@justbook/Layer-Depth-Attention-Kaggle-103/runs/htv1zdoxfxfr5seokcrsc

{'shared_kv_depth_memory_dualq_sublayer': {'method': 'shared_kv_depth_memory_dualq_sublayer',
  'model_params': 33889152,
  'best_step': 40000,
  'best_val_loss': 4.3656229758262635,
  'best_val_ppl': 78.69841188173254,
  'best_test_loss': 4.390099545832945,
  'best_test_ppl': 80.64844679771473,
  'history': [{'step': 1,
    'lr': 2.9999999999999997e-06,
    'train_loss_raw': 10.883245468139648,
    'train_loss_avg': 10.883245468139648,
    'val_loss': 10.855297346115112,
    'val_ppl': 51807.8699433158,
    'elapsed_seconds': 7.875338900000003,
    'elapsed_minutes': 0.1312556483333334},
   {'step': 1000,
    'lr': 0.0002996611862696745,
    'train_loss_raw': 5.944774150848389,
    'train_loss_avg': 6.03761920928955,
    'val_loss': 6.0304656457901,
    'val_ppl': 415.90865041120907,
    'elapsed_seconds': 124.7874127,
    'elapsed_minutes': 2.079790211666667},
   {'step': 2000,
    'lr': 0.0002984921615403923,
    'train_loss_raw': 5.796938896179199,
    'train_loss_avg': 5.844565653

In [10]:
import pandas as pd

rows = []
for name, summary in results.items():
    rows.append({
        "method": name,
        "model_params": summary["model_params"],
        "best_step": summary["best_step"],
        "best_val_loss": summary["best_val_loss"],
        "best_val_ppl": summary["best_val_ppl"],
        "best_test_loss": summary["best_test_loss"],
        "best_test_ppl": summary["best_test_ppl"],
    })

df = pd.DataFrame(rows).sort_values("best_test_ppl")
display(df)

for name, summary in results.items():
    print("=" * 100)
    print("last history rows:", name)
    print("=" * 100)
    hist_df = pd.DataFrame(summary["history"])
    display(hist_df.tail(10))


,method,model_params,best_step,best_val_loss,best_val_ppl,best_test_loss,best_test_ppl
0,shared_kv_depth_memory_dualq_sublayer,33889152,40000,4.365623,78.698412,4.3901,80.648447


last history rows: shared_kv_depth_memory_dualq_sublayer


,step,lr,train_loss_raw,train_loss_avg,val_loss,val_ppl,elapsed_seconds,elapsed_minutes
31,31000,0.000063,4.613888,4.577911,4.477241,87.991553,3596.061663,59.934361
32,32000,0.000056,4.511815,4.553583,4.431830,84.085150,3711.528711,61.858812
33,33000,0.000050,4.397692,4.506931,4.404956,81.855514,3827.154639,63.785911
34,34000,0.000045,4.675789,4.568707,4.431506,84.057913,3943.004018,65.716734
35,35000,0.000040,4.809904,4.580092,4.398018,81.289609,4058.673463,67.644558
36,36000,0.000037,4.747350,4.488697,4.412043,82.437681,4174.381109,69.573018
37,37000,0.000034,4.526864,4.640790,4.421330,83.206918,4289.998887,71.499981
38,38000,0.000032,5.155404,4.562471,4.367071,78.812483,4405.650893,73.427515
39,39000,0.000030,4.359382,4.494183,4.405164,81.872571,4521.148425,75.352474
40,40000,0.000030,4.757684,4.573398,4.365623,78.698412,4636.839447,77.280657
